# 01 - Cleaning: IBM Employee Attrition Dataset
Central command notebook. Cleans raw_dirty.csv, saves clean_master.csv, then splits into train/test/present.

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../src')
from utils import basic_report, TARGET

df = pd.read_csv('../data/raw_dirty.csv')
basic_report(df)
df.head()


## 1. Fix column names & whitespace

In [ ]:
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()


## 2. Drop exact duplicate rows

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"dropped {before - len(df)} duplicate rows")


## 3. Standardize inconsistent categorical casing/labels

In [ ]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.title()

# common attrition dataset fixes - adjust to whatever dirt you introduced
if TARGET in df.columns:
    df[TARGET] = df[TARGET].replace({'Y': 'Yes', 'N': 'No', '1': 'Yes', '0': 'No'})


## 4. Handle missing values

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include='object').columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isnull().sum().sum()  # should be 0


## 5. Fix dtypes (e.g. numbers stored as strings)

In [ ]:
for col in df.columns:
    if df[col].dtype == 'object':
        converted = pd.to_numeric(df[col], errors='coerce')
        if converted.notnull().mean() > 0.95:  # mostly numeric -> convert
            df[col] = converted.fillna(converted.median())


## 6. Outlier check (cap using IQR)

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df[col] = df[col].clip(lower, upper)


## 7. Final check & save clean_master.csv

In [ ]:
basic_report(df)
df.to_csv('../data/clean_master.csv', index=False)


## 8. Split into train / test / present
Present set drops the target column - simulates unseen new employees.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df[TARGET] if TARGET in df else None)
test_df, present_df = train_test_split(temp_df, test_size=0.5, random_state=42)

present_df = present_df.drop(columns=[TARGET]) if TARGET in present_df.columns else present_df

train_df.to_csv('../data/train.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)
present_df.to_csv('../data/present.csv', index=False)

print(train_df.shape, test_df.shape, present_df.shape)
